# Prompt A - London House Price Prediction

This notebook was created for the benchmark run in `benchmark/codex/run1`.

Run status for this session:
- Required input files `london_house_prices.csv` and `london_area_features.csv` were not present anywhere in the workspace.
- The checked-in virtual environment was also broken because its base Python interpreter was unavailable.
- The notebook therefore includes a complete, reproducible pipeline, but it could not be executed end-to-end in this session without the missing CSV files.

## Setup

This notebook assumes both CSV files are placed next to the notebook inside `benchmark/codex/run1/`.
All stochastic steps use `random_state=42`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / 'benchmark' / 'codex' / 'run1',
]

base_dir = next((path for path in candidate_dirs if (path / 'london_house_prices.csv').exists() or (path / 'london_area_features.csv').exists()), candidate_dirs[-1])
prices_path = base_dir / 'london_house_prices.csv'
areas_path = base_dir / 'london_area_features.csv'

print(f'Working directory: {base_dir}')
print(f'Prices path exists: {prices_path.exists()} -> {prices_path.name}')
print(f'Areas path exists: {areas_path.exists()} -> {areas_path.name}')

missing_files = [str(path.name) for path in [prices_path, areas_path] if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing required input files: ' + ', '.join(missing_files) +
        '. Place both CSV files in the same folder as this notebook before running.'
    )

## Stage 1 - Data Ingestion and Quality Assessment

In [ ]:
prices = pd.read_csv(prices_path)
areas = pd.read_csv(areas_path)

print('Prices shape:', prices.shape)
print('Areas shape:', areas.shape)

print('\nPrices columns:')
print(prices.columns.tolist())
print('\nAreas columns:')
print(areas.columns.tolist())

print('\nPrices dtypes:')
print(prices.dtypes)
print('\nAreas dtypes:')
print(areas.dtypes)

In [ ]:
def missing_report(df, name):
    report = (
        df.isna()
        .sum()
        .rename('missing_count')
        .to_frame()
        .assign(missing_pct=lambda x: x['missing_count'] / len(df) * 100)
        .query('missing_count > 0')
        .sort_values(['missing_pct', 'missing_count'], ascending=False)
    )
    print(f'\nMissing value report for {name}:')
    display(report if not report.empty else pd.DataFrame({'message': ['No missing values found']}))
    return report

prices_missing = missing_report(prices, 'prices')
areas_missing = missing_report(areas, 'areas')

price_quantiles = prices['price'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
q1 = prices['price'].quantile(0.25)
q3 = prices['price'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_mask = (prices['price'] < lower_bound) | (prices['price'] > upper_bound)

print('\nPrice quantiles:')
print(price_quantiles)
print(f'\nIQR outlier bounds: ({lower_bound:,.0f}, {upper_bound:,.0f})')
print(f'Outlier rows by IQR rule: {outlier_mask.sum():,} ({outlier_mask.mean() * 100:.2f}%)')

print('\nRecommended handling: keep records for baseline analysis, evaluate log(price), and compare with winsorised or capped variants in Stage 4 rather than deleting high-end London properties outright.')

prices['outcode'] = prices['outcode'].astype(str).str.strip().str.upper()
areas['outcode'] = areas['outcode'].astype(str).str.strip().str.upper()

duplicate_outcodes = areas['outcode'].duplicated().sum()
print(f'\nDuplicate outcodes in area table: {duplicate_outcodes}')
if duplicate_outcodes:
    areas = areas.drop_duplicates(subset='outcode', keep='first').copy()
    print('Dropped duplicate area rows by outcode, keeping first occurrence.')

merged = prices.merge(areas, on='outcode', how='left', validate='many_to_one', indicator=True)
print('\nMerged shape:', merged.shape)
print('Merge indicator counts:')
print(merged['_merge'].value_counts(dropna=False))

unmatched_outcodes = merged.loc[merged['_merge'] != 'both', 'outcode'].nunique()
print(f'Unique unmatched property outcodes: {unmatched_outcodes}')

numeric_cols = merged.select_dtypes(include=np.number).columns.tolist()
categorical_cols = merged.select_dtypes(exclude=np.number).columns.tolist()

print('\nNumeric columns after merge:', len(numeric_cols))
print('Categorical columns after merge:', len(categorical_cols))

## Stage 2 - Exploratory Data Analysis

In [ ]:
plot_dir = base_dir / 'plots'
plot_dir.mkdir(exist_ok=True)
saved_plots = []

print('Stage 2 dataset shape:', merged.shape)

price_summary = merged['price'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
price_skew = merged['price'].skew()
log_price_skew = np.log1p(merged['price']).skew()
print('\nPrice summary statistics:')
display(price_summary.to_frame('price'))
print(f'Raw price skewness: {price_skew:.3f}')
print(f'Log(price) skewness: {log_price_skew:.3f}')
if price_skew > 1:
    print('Comment: House prices are strongly right-skewed, which is expected for property values. A log transform should make modelling easier and reduce the influence of extreme values.')
else:
    print('Comment: House prices are not heavily skewed in this sample, but a log transform may still improve model stability.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(merged['price'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Distribution of House Prices')
sns.histplot(np.log1p(merged['price']), bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Distribution of log(1 + Price)')
plt.tight_layout()
distribution_plot = plot_dir / 'price_distribution.png'
plt.savefig(distribution_plot, dpi=200)
saved_plots.append(distribution_plot.name)
plt.show()

numeric_corr = merged[numeric_cols].corr(numeric_only=True)
top_numeric_features = numeric_corr['price'].drop('price').abs().sort_values(ascending=False).head(11).index.tolist()
heatmap_features = ['price'] + top_numeric_features

print('\nTop numeric features by absolute correlation with price:')
display(numeric_corr.loc[heatmap_features, ['price']].sort_values('price', key=lambda s: s.abs(), ascending=False))

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_corr.loc[heatmap_features, heatmap_features], cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap of Top Numeric Features')
plt.tight_layout()
heatmap_plot = plot_dir / 'top_numeric_correlation_heatmap.png'
plt.savefig(heatmap_plot, dpi=200)
saved_plots.append(heatmap_plot.name)
plt.show()

property_type_summary = (
    merged.groupby('propertyType')['price']
    .agg(['count', 'median', 'mean'])
    .sort_values('median', ascending=False)
)
print('\nPrice summary by property type:')
display(property_type_summary)

plt.figure(figsize=(12, 6))
sns.boxplot(data=merged, x='propertyType', y='price', order=property_type_summary.index)
plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.title('Prices by Property Type (log scale)')
plt.ylabel('Price (log scale)')
plt.tight_layout()
property_type_plot = plot_dir / 'price_by_property_type.png'
plt.savefig(property_type_plot, dpi=200)
saved_plots.append(property_type_plot.name)
plt.show()

area_numeric_features = [
    col for col in areas.columns
    if col != 'outcode' and pd.api.types.is_numeric_dtype(areas[col]) and col in merged.columns
]

area_feature_correlations = (
    merged[area_numeric_features + ['price']]
    .corr(numeric_only=True)['price']
    .drop('price')
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
selected_area_features = area_feature_correlations.head(3).index.tolist()

print('\nTop area-level features by absolute correlation with price:')
display(area_feature_correlations.head(10).to_frame('corr_with_price'))

sample_size = min(len(merged), 15000)
eda_sample = merged.sample(sample_size, random_state=RANDOM_STATE)

for feature in selected_area_features:
    plt.figure(figsize=(8, 6))
    sns.regplot(data=eda_sample, x=feature, y='price', scatter_kws={'alpha': 0.25, 's': 18}, line_kws={'color': 'crimson'})
    plt.title(f'Price vs {feature}')
    plt.tight_layout()
    feature_plot = plot_dir / f'price_vs_{feature}.png'
    plt.savefig(feature_plot, dpi=200)
    saved_plots.append(feature_plot.name)
    plt.show()

top_property_type = property_type_summary.index[0]
bottom_property_type = property_type_summary.index[-1]
top_numeric_driver = numeric_corr['price'].drop('price').abs().idxmax()
top_area_driver = area_feature_correlations.abs().idxmax()

insights = [
    f"Price is strongly right-skewed (skew={price_skew:.2f}), while log(price) is much closer to symmetric (skew={log_price_skew:.2f}), so log-transforming the target is likely to improve model fit.",
    f"The strongest numeric predictor in the merged data is '{top_numeric_driver}', suggesting that structural property characteristics dominate price variation.",
    f"Median prices vary materially across property types: '{top_property_type}' has the highest median price and '{bottom_property_type}' the lowest, while the most informative area-level feature is '{top_area_driver}'.",
]

print('\nThree most important Stage 2 insights:')
for idx, insight in enumerate(insights, start=1):
    print(f'{idx}. {insight}')

print('\nSaved plot files:')
for plot_name in saved_plots:
    print(f'- {plot_name}')

## Stage 3 - Baseline Models

In [ ]:
plot_dir = base_dir / 'plots'
plot_dir.mkdir(exist_ok=True)
stage3_saved_plots = []

model_data = merged.drop(columns=['_merge']).copy()
model_data = model_data[model_data['price'].notna()].copy()

X = model_data.drop(columns=['price'])
y = model_data['price']

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Stage 3 modelling dataset shape:', model_data.shape)
print(f'Number of numeric features: {len(numeric_features)}')
print(f'Number of categorical features: {len(categorical_features)}')
print('\nMissing values in target:', int(y.isna().sum()))

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

print('\nPreprocessing strategy:')
print('- Numeric features: median imputation + standard scaling')
print('- Categorical features: most-frequent imputation + one-hot encoding')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('\nTrain/test split:')
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

results = []
predictions = {}
baseline_pipelines = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    baseline_pipelines[name] = pipeline
    predictions[name] = preds
    results.append({
        'model': name,
        'rmse': mean_squared_error(y_test, preds, squared=False),
        'mae': mean_absolute_error(y_test, preds),
        'r2': r2_score(y_test, preds),
    })

baseline_results = pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)
print('\nBaseline model performance on the test set:')
display(baseline_results.style.format({'rmse': '{:,.2f}', 'mae': '{:,.2f}', 'r2': '{:.4f}'}))

for name, preds in predictions.items():
    plt.figure(figsize=(7, 7))
    sns.scatterplot(x=y_test, y=preds, alpha=0.35)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    plt.plot(lims, lims, linestyle='--', color='black')
    plt.xlabel('Actual Price')
    plt.ylabel('Predicted Price')
    plt.title(f'Predicted vs Actual - {name}')
    plt.tight_layout()
    plot_name = f'predicted_vs_actual_{name.lower().replace(" ", "_")}.png'
    plt.savefig(plot_dir / plot_name, dpi=200)
    stage3_saved_plots.append(plot_name)
    plt.show()

best_baseline = baseline_results.iloc[0]
worst_baseline = baseline_results.iloc[-1]

print('\nModel comparison:')
print(
    f"{best_baseline['model']} performs better on the test set because it has the lowest RMSE ({best_baseline['rmse']:,.2f}), "
    f"the lowest MAE ({best_baseline['mae']:,.2f}), and the highest R^2 ({best_baseline['r2']:.4f})."
)
print(
    'This pattern is expected when property prices depend on non-linear interactions and threshold effects that tree-based models can capture better than a purely linear baseline.'
)
print(
    f"For reference, the weaker baseline here is {worst_baseline['model']}."
)

print('\nStage 3 saved plot files:')
for plot_name in stage3_saved_plots:
    print(f'- {plot_name}')

## Stage 4 - Performance Improvement

In [ ]:
plot_dir = base_dir / 'plots'
plot_dir.mkdir(exist_ok=True)
stage4_saved_plots = []

print('Stage 4 improvement strategies to evaluate:')
print('1. Log-transform the target and train a Gradient Boosting regressor')
print('2. Add polynomial interaction terms for key structural numeric features')
print('3. Tune a Random Forest with capped training targets to reduce the effect of extreme prices')

def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test, log_target=False):
    target_train = np.log1p(y_train) if log_target else y_train
    pipeline.fit(X_train, target_train)
    preds = pipeline.predict(X_test)
    if log_target:
        preds = np.expm1(preds)
    metrics = {
        'model': name,
        'rmse': mean_squared_error(y_test, preds, squared=False),
        'mae': mean_absolute_error(y_test, preds),
        'r2': r2_score(y_test, preds),
    }
    return metrics, preds, pipeline

train_price_cap = y_train.quantile(0.99)
y_train_capped = y_train.clip(upper=train_price_cap)
print(f'\n99th percentile cap applied to y_train for one strategy: {train_price_cap:,.2f}')

log_gbr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=RANDOM_STATE)),
])

poly_numeric = [
    col for col in numeric_features
    if col in {'floorAreaSqM', 'bedrooms', 'bathrooms', 'livingRooms', 'rentEstimate', 'latitude', 'longitude'}
]
remaining_numeric = [col for col in numeric_features if col not in poly_numeric]

poly_preprocessor = ColumnTransformer(
    transformers=[
        ('poly_num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('scaler', StandardScaler()),
        ]), poly_numeric),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), remaining_numeric),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

poly_linear = Pipeline([
    ('preprocessor', poly_preprocessor),
    ('model', LinearRegression()),
])

rf_search_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

search = RandomizedSearchCV(
    estimator=rf_search_pipeline,
    param_distributions={
        'model__n_estimators': [200, 300],
        'model__max_depth': [None, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
    },
    n_iter=6,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

improved_results = []
improved_predictions = {}
improved_pipelines = {}

result, log_gbr_preds, log_gbr_pipeline = evaluate_model(
    'Gradient Boosting (log target)', log_gbr, X_train, X_test, y_train, y_test, log_target=True
)
improved_results.append(result)
improved_predictions['Gradient Boosting (log target)'] = log_gbr_preds
improved_pipelines['Gradient Boosting (log target)'] = log_gbr_pipeline

result, poly_preds, poly_pipeline = evaluate_model(
    'Linear Regression + Polynomial Features', poly_linear, X_train, X_test, y_train, y_test, log_target=False
)
improved_results.append(result)
improved_predictions['Linear Regression + Polynomial Features'] = poly_preds
improved_pipelines['Linear Regression + Polynomial Features'] = poly_pipeline

search.fit(X_train, y_train_capped)
rf_tuned_preds = search.best_estimator_.predict(X_test)
rf_tuned_metrics = {
    'model': 'Random Forest Tuned (99th percentile cap on y_train)',
    'rmse': mean_squared_error(y_test, rf_tuned_preds, squared=False),
    'mae': mean_absolute_error(y_test, rf_tuned_preds),
    'r2': r2_score(y_test, rf_tuned_preds),
}
improved_results.append(rf_tuned_metrics)
improved_predictions['Random Forest Tuned (99th percentile cap on y_train)'] = rf_tuned_preds
improved_pipelines['Random Forest Tuned (99th percentile cap on y_train)'] = search.best_estimator_

print('\nBest Random Forest hyperparameters from RandomizedSearchCV:')
print(search.best_params_)

all_results = pd.concat([
    baseline_results,
    pd.DataFrame(improved_results),
], ignore_index=True).sort_values('rmse').reset_index(drop=True)

print('\nAll model results sorted by RMSE:')
display(all_results.style.format({'rmse': '{:,.2f}', 'mae': '{:,.2f}', 'r2': '{:.4f}'}))

best_model_name = all_results.iloc[0]['model']
if best_model_name in improved_pipelines:
    best_model = improved_pipelines[best_model_name]
    best_model_preds = improved_predictions[best_model_name]
else:
    best_model = baseline_pipelines[best_model_name]
    best_model_preds = predictions[best_model_name]

plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=best_model_preds, alpha=0.35)
lims = [min(y_test.min(), best_model_preds.min()), max(y_test.max(), best_model_preds.max())]
plt.plot(lims, lims, linestyle='--', color='black')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Predicted vs Actual - Best Model ({best_model_name})')
plt.tight_layout()
best_model_plot = plot_dir / 'predicted_vs_actual_best_model.png'
plt.savefig(best_model_plot, dpi=200)
stage4_saved_plots.append(best_model_plot.name)
plt.show()

feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
best_estimator = best_model.named_steps['model']
if hasattr(best_estimator, 'feature_importances_'):
    raw_importance = pd.Series(best_estimator.feature_importances_, index=feature_names)
elif hasattr(best_estimator, 'coef_'):
    coef_values = np.ravel(best_estimator.coef_)
    raw_importance = pd.Series(np.abs(coef_values), index=feature_names)
else:
    raise AttributeError('Best model does not expose feature importances or coefficients.')

feature_importances = raw_importance.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
feature_importances.sort_values().plot(kind='barh')
plt.title(f'Top 20 Feature Importances - {best_model_name}')
plt.tight_layout()
feature_importance_plot = plot_dir / 'best_model_feature_importances.png'
plt.savefig(feature_importance_plot, dpi=200)
stage4_saved_plots.append(feature_importance_plot.name)
plt.show()

print('\nTop 20 feature importances from the best model:')
display(feature_importances.to_frame('importance'))

top_features = feature_importances.head(5).index.tolist()
print('\nInterpretation of what drives London house prices:')
print(
    f"The best-performing model is {best_model_name}, which suggests that non-linear structure and interactions matter beyond the simple baselines."
)
print(
    'The strongest drivers are concentrated in a small set of structural and location-linked features, especially those related to size, room counts, rent proxy variables, and area context.'
)
print(
    f"The top ranked features in this run are: {', '.join(top_features)}. These features are the main candidates for explaining price differences across London listings."
)

print('\nStage 4 saved plot files:')
for plot_name in stage4_saved_plots:
    print(f'- {plot_name}')